# Lab 2 — Task 2: Music Genre Classification with VGG16

Notebook independiente para la **Task 2**, sin modificar el notebook principal. Utiliza transferencia de aprendizaje con VGG16 preentrenada en ImageNet.

Pasos: cargar espectrogramas, separar datos, congelar VGG16, entrenar una cabeza de clasificación y evaluar el modelo seleccionado.

**Criterio del enunciado:** se reserva el 20 % para test desde el principio. El 80 % restante se utiliza para entrenar y seleccionar la mejor época mediante validación. El test no interviene en la optimización.

## 1. Imports y configuración

TensorFlow/Keras implementa VGG16. Si faltan librerías, descomenta y ejecuta la instalación.

In [ ]:
# %pip install tensorflow scikit-learn matplotlib
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Fijar las semillas hace más reproducible la partición y la inicialización del modelo.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)

## 2. Cargar imágenes y dividir los datos

Cambia `DATA_DIR` para que apunte a `images_original`. La división inicial reserva el 20 % para test. La validación se extrae únicamente del 80 % restante.

In [ ]:
# Ejemplo en Colab: '/content/drive/MyDrive/GTZAN/images_original'
DATA_DIR = 'Data/images_original'  # Cambia la ruta si guardaste los datos en otro lugar.
IMG_SIZE = (224, 224)               # Tamaño requerido por VGG16.
BATCH_SIZE = 32

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"No encuentro {DATA_DIR}. Cambia DATA_DIR a images_original.")

# Cada subcarpeta representa un género musical.
class_names = sorted([d for d in os.listdir(DATA_DIR)
                      if os.path.isdir(os.path.join(DATA_DIR, d))])
extensions = ('.png', '.jpg', '.jpeg', '.bmp')
paths, labels = [], []
for label, genre in enumerate(class_names):
    folder = os.path.join(DATA_DIR, genre)
    for filename in sorted(os.listdir(folder)):
        if filename.lower().endswith(extensions):
            paths.append(os.path.join(folder, filename))
            labels.append(label)

paths, labels = np.array(paths), np.array(labels)
if len(paths) == 0:
    raise ValueError("No se han encontrado imágenes. Comprueba la estructura de carpetas.")

print("Géneros:", class_names)
print("Total de imágenes:", len(paths))

# División solicitada: 80 % para desarrollo y 20 % reservado para evaluación final.
# stratify mantiene la proporción de cada género en ambos conjuntos.
dev_paths, test_paths, dev_labels, test_labels = train_test_split(
    paths, labels, test_size=0.20, random_state=SEED, stratify=labels
)

# La validación se extrae del 80 % de desarrollo, nunca del conjunto de test.
train_paths, val_paths, train_labels, val_labels = train_test_split(
    dev_paths, dev_labels, test_size=0.20, random_state=SEED, stratify=dev_labels
)
print(f"Train: {len(train_paths)} | Validation: {len(val_paths)} | Test: {len(test_paths)}")
print(f"Porcentajes: train={len(train_paths)/len(paths):.1%}, "
      f"validation={len(val_paths)/len(paths):.1%}, test={len(test_paths)/len(paths):.1%}")

## 3. Preparar los datasets

Las imágenes se decodifican como RGB y se redimensionan a 224 × 224. Se mezclan solo los datos de entrenamiento; validación y test conservan su orden para poder comparar las predicciones con las etiquetas originales.

In [ ]:
def load_image(path, label):
    # Lee el archivo y decodifica la imagen con tres canales de color.
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    # VGG16 requiere entradas de 224 x 224 píxeles.
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

def make_dataset(file_paths, file_labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, file_labels))
    if training:
        # Mezclar el orden de entrenamiento ayuda a evitar que el orden de archivos influya.
        ds = ds.shuffle(len(file_paths), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    # Los lotes y prefetch hacen más eficiente la carga de datos.
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels)
test_ds = make_dataset(test_paths, test_labels)

# Mostrar algunos ejemplos sirve para comprobar que las etiquetas y las imágenes son correctas.
for images, labels_batch in train_ds.take(1):
    plt.figure(figsize=(10, 3))
    for i in range(min(5, len(images))):
        plt.subplot(1, 5, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(class_names[int(labels_batch[i])])
        plt.axis('off')
    plt.tight_layout()
    plt.show()

## 4. Cargar VGG16 preentrenada

Se eliminan las capas finales de ImageNet porque clasifican 1000 categorías distintas. Congelamos las capas convolucionales para reutilizar sus características visuales sin reentrenar todo el modelo, lo que reduce el coste computacional.

In [ ]:
# include_top=False elimina la cabeza de clasificación original de ImageNet.
base = tf.keras.applications.VGG16(
    weights='imagenet', include_top=False, input_shape=(224, 224, 3)
)

# Congelar VGG16 mantiene fijos los pesos preentrenados durante esta tarea.
base.trainable = False
base.summary()

## 5. Añadir una cabeza para los géneros musicales

VGG16 necesita su función de preprocesamiento específica. GlobalAveragePooling2D resume los mapas de características con menos parámetros que Flatten. La regularización L2 y Dropout ayudan a reducir el sobreajuste de la cabeza de clasificación.

In [ ]:
# La entrada se mantiene en el rango 0–255 hasta aplicar preprocess_input.
inputs = tf.keras.Input(shape=(224, 224, 3))

# Aumento de datos ligero: pequeñas transformaciones para reducir el sobreajuste.
# Estas capas aplican transformaciones aleatorias durante el entrenamiento.
x = tf.keras.layers.RandomTranslation(0.05, 0.05)(inputs)
x = tf.keras.layers.RandomZoom(0.1)(x)
x = tf.keras.layers.RandomContrast(0.1)(x)

# Convierte los píxeles al formato esperado por los pesos preentrenados de VGG16.
x = tf.keras.applications.vgg16.preprocess_input(x)

# Extrae características visuales con VGG16 congelada.
x = base(x, training=False)

# Resume cada mapa espacial y reduce la cantidad de parámetros entrenables.
x = tf.keras.layers.GlobalAveragePooling2D()(x)

# La cabeza aprende qué características distinguen los géneros musicales.
x = tf.keras.layers.Dense(
    256, activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(1e-4)  # Penaliza pesos grandes.
)(x)

# Dropout desactiva parte de las activaciones durante el entrenamiento para regularizar.
x = tf.keras.layers.Dropout(0.4)(x)

# Softmax produce una probabilidad por cada género.
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
vgg_model = tf.keras.Model(inputs, outputs)
vgg_model.summary()

## 6. Entrenar el modelo

Adam usa un learning rate pequeño para entrenar la cabeza sin alterar el extractor preentrenado. EarlyStopping vigila la accuracy de validación y restaura los pesos de la mejor época. Se pueden ejecutar hasta 15 épocas, pero el entrenamiento puede detenerse antes.

In [ ]:
vgg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',  # Las etiquetas son números enteros.
    metrics=['accuracy']
)

# Detener si val_accuracy no mejora durante 3 épocas y recuperar los mejores pesos.
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', mode='max', patience=3, restore_best_weights=True
)

vgg_history = vgg_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    shuffle=False,  # train_ds ya se mezcla mediante Dataset.shuffle().
    callbacks=[early_stop]
)

best_val_accuracy = max(vgg_history.history['val_accuracy'])
print(f"Mejor validation accuracy: {best_val_accuracy:.4f}")

## 7. Curvas de entrenamiento

La accuracy muestra qué proporción de ejemplos se clasifica correctamente. La loss refleja el error de las probabilidades del modelo. Comparar entrenamiento y validación ayuda a detectar sobreajuste.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(vgg_history.history['accuracy'], marker='o', label='Train')
axes[0].plot(vgg_history.history['val_accuracy'], marker='o', label='Validation')
axes[0].set(title='VGG16 — Accuracy', xlabel='Época', ylabel='Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(vgg_history.history['loss'], marker='o', label='Train')
axes[1].plot(vgg_history.history['val_loss'], marker='o', label='Validation')
axes[1].set(title='VGG16 — Loss', xlabel='Época', ylabel='Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 8. Evaluación final sobre test

Una vez seleccionados los pesos con validación, evaluamos el modelo sobre el 20 % reservado. Este conjunto no se utiliza para entrenar ni seleccionar la mejor época, siguiendo el enunciado.

In [ ]:
# Evaluación final sobre el conjunto de test reservado.
test_loss, test_accuracy = vgg_model.evaluate(test_ds, verbose=0)
print(f"VGG16 — test loss: {test_loss:.4f}")
print(f"VGG16 — test accuracy: {test_accuracy:.4f}")

# Obtener las clases predichas para calcular métricas por género.
# test_ds no se mezcla, por lo que el orden coincide con test_labels.
y_true = test_labels
y_prob = vgg_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("\nInforme de clasificación en test:")
print(classification_report(
    y_true, y_pred, labels=range(len(class_names)),
    target_names=class_names, zero_division=0
))

# La matriz de confusión muestra qué géneros se confunden entre sí.
cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    xticks_rotation=45, cmap='Blues', colorbar=False
)
plt.title('VGG16 — Matriz de confusión (test)')
plt.tight_layout()
plt.show()

## 9. Conclusión y notas

Anota la validation accuracy y la test accuracy después de ejecutar el notebook. Compara la test accuracy con Task 1, asegurándote de que ambas usan la misma partición.

- La primera ejecución puede descargar los pesos de VGG16 de ImageNet.
- Si no hay GPU disponible, el notebook funciona en CPU, pero puede tardar bastante.
- Si existen varias imágenes por canción, la evaluación más rigurosa consiste en separar por canción original para evitar que imágenes relacionadas aparezcan en train y test.